# Conflict Analysis

Investigates conflicts in the mapping rulebook — descriptions mapped to more than one category.

**Run order:**
1. Setup cell (always run first)
2. Section 1 — find duplicates in the v2 rulebook
3. Section 2 — trace each duplicate back to its source xlsm file
4. Section 3 — export results to two-tab Excel report
5. Section 4 — test whether a rulebook-first row detection approach is viable

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / 'config' / 'paths.py').exists() else _cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from config.paths import (
    RULEBOOK_DB_V2_PATH,
    RULEBOOK_DB_PATH,
    ANALYSIS_OUTPUT_DIR,
    BLOCK_DETECTION_DIR,
    MAPPED_T12_DOCS_DIR,
    NEW_MAPPED_T12_DOCS_DIR,
    CATEGORY_CLASSIFICATION_PATH,
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Analysis output dir: {ANALYSIS_OUTPUT_DIR}')

## Section 1 — Find Rulebook Duplicates

Finds all row descriptions that appear in more than one category in the active v2 rulebook.
These are ambiguous entries — the same vendor text was mapped to different canonical categories
across source files.

**Output:** `output_docs/analysis/rulebook_duplicates_v2.json`

In [ ]:
OUTPUT_PATH = ANALYSIS_OUTPUT_DIR / 'rulebook_duplicates_v2.json'

with open(RULEBOOK_DB_V2_PATH) as f:
    rulebook = json.load(f)

desc_to_categories = defaultdict(list)
for category, descriptions in rulebook.items():
    for desc in descriptions:
        desc_normalized = desc.strip().lower()
        desc_to_categories[desc_normalized].append({
            'category': category,
            'original_desc': desc,
        })

duplicates = {
    desc: entries
    for desc, entries in desc_to_categories.items()
    if len(entries) > 1
}

print(f"Rulebook: {len(rulebook)} categories, "
      f"{sum(len(v) for v in rulebook.values())} total description entries")
print(f"Duplicate descriptions (appear in 2+ categories): {len(duplicates)}\n")

if not duplicates:
    print('No duplicates found.')
else:
    print('=' * 70)
    for i, (desc_lower, entries) in enumerate(sorted(duplicates.items()), 1):
        cats = [e['category'] for e in entries]
        original = entries[0]['original_desc']
        print(f"{i:3d}.  '{original}'")
        for e in entries:
            marker = '  <-' if e['category'] != cats[0] else '   '
            print(f"       {marker}  category: '{e['category']}'")
        print()

    print('=' * 70)
    print('SUMMARY — which category pairs have conflicts:\n')
    pair_counts = defaultdict(list)
    for desc_lower, entries in duplicates.items():
        cats = tuple(sorted(set(e['category'] for e in entries)))
        pair_counts[cats].append(entries[0]['original_desc'])

    for pair, descs in sorted(pair_counts.items(), key=lambda x: -len(x[1])):
        print(f"  {pair[0]}  <->  {pair[1] if len(pair) > 1 else '(self)'}")
        print(f"  {len(descs)} conflicting description(s):")
        for d in descs:
            print(f"    - '{d}'")
        print()

In [ ]:
pair_counts_for_json = defaultdict(list)
for desc_lower, entries in duplicates.items():
    cats = tuple(sorted(set(e['category'] for e in entries)))
    pair_counts_for_json[cats].append(entries[0]['original_desc'])

output_data = {
    'summary': {
        'total_categories': len(rulebook),
        'total_descriptions': sum(len(v) for v in rulebook.values()),
        'total_duplicates': len(duplicates),
    },
    'duplicates': [
        {
            'description': entries[0]['original_desc'],
            'description_normalized': desc_lower,
            'categories': [e['category'] for e in entries],
        }
        for desc_lower, entries in sorted(duplicates.items())
    ],
    'conflicts_by_category_pair': [
        {
            'category_a': pair[0],
            'category_b': pair[1] if len(pair) > 1 else pair[0],
            'conflict_count': len(descs),
            'descriptions': sorted(descs),
        }
        for pair, descs in sorted(pair_counts_for_json.items(), key=lambda x: -len(x[1]))
    ],
}

with open(OUTPUT_PATH, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f'Saved: {OUTPUT_PATH}')

## Section 2 — Trace Conflict Sources to Files

For every duplicate description, scans all 19 human-reviewed xlsm source files to find which
files contributed each conflicting mapping. Also flags cross-bucket conflicts (REVENUE vs EXPENSE).

**Input:** `output_docs/analysis/rulebook_duplicates.json` (v1 duplicates — has 94 conflicts)

**Output:** `output_docs/analysis/conflict_file_sources_v2.json`

In [ ]:
DUPLICATES_PATH = ANALYSIS_OUTPUT_DIR / 'rulebook_duplicates.json'
OUTPUT_JSON = ANALYSIS_OUTPUT_DIR / 'conflict_file_sources_v2.json'

with open(DUPLICATES_PATH) as f:
    dup_data = json.load(f)

dup_lookup = {
    d['description_normalized']: d['categories']
    for d in dup_data['duplicates']
}

print(f'Loaded {len(dup_lookup)} duplicate descriptions to search for.\n')

with open(CATEGORY_CLASSIFICATION_PATH) as f:
    classification = json.load(f)

def get_bucket(category):
    return classification.get(category, 'UNKNOWN')

In [ ]:
xlsm_files = (
    sorted(MAPPED_T12_DOCS_DIR.glob('*.xlsm')) +
    sorted(NEW_MAPPED_T12_DOCS_DIR.glob('*.xlsm'))
)
xlsm_files = [f for f in xlsm_files if not f.name.startswith('~$')]

print(f'Scanning {len(xlsm_files)} xlsm files...\n')

desc_file_hits = defaultdict(list)

for fpath in xlsm_files:
    try:
        wb = openpyxl.load_workbook(fpath, data_only=True, read_only=True)
        if 'Financials' not in wb.sheetnames:
            wb.close()
            continue
        ws = wb['Financials']

        anchor_row = None
        for row in ws.iter_rows():
            for cell in row:
                if cell.value and isinstance(cell.value, str) and 'COMPLETE MAPPING HERE' in cell.value.upper():
                    anchor_row = cell.row
                    break
            if anchor_row:
                break

        if not anchor_row:
            wb.close()
            continue

        all_rows = list(ws.iter_rows(min_row=anchor_row + 1, values_only=True))
        wb.close()

        if not all_rows:
            continue

        headers = all_rows[0]
        try:
            desc_col = headers.index('Row Description (from T12)')
            fm_col   = headers.index('Final_Mapping')
        except ValueError:
            continue

        for data_row in all_rows[1:]:
            if not data_row or data_row[0] is None:
                break
            desc = data_row[desc_col] if desc_col < len(data_row) else None
            if not desc or not isinstance(desc, str):
                continue
            desc_norm = desc.strip().lower()
            if desc_norm not in dup_lookup:
                continue
            fm = data_row[fm_col] if fm_col < len(data_row) else None
            if not fm or not isinstance(fm, str):
                continue
            desc_file_hits[desc_norm].append({
                'file': fpath.name,
                'final_mapping': fm.strip().title(),
            })

    except Exception as e:
        print(f'  ERROR reading {fpath.name}: {e}')

In [ ]:
found_in_files = []
not_found = []

for desc_norm, conflict_cats in sorted(dup_lookup.items()):
    hits = desc_file_hits.get(desc_norm, [])

    if not hits:
        not_found.append(desc_norm)
        continue

    by_mapping = defaultdict(list)
    for h in hits:
        by_mapping[h['final_mapping']].append(h['file'])

    cat_buckets = {cat: get_bucket(cat) for cat in conflict_cats}
    unique_buckets = set(cat_buckets.values())
    cross_bucket = 'REVENUE' in unique_buckets and 'EXPENSE' in unique_buckets

    found_in_files.append({
        'description': next(
            d['description'] for d in dup_data['duplicates']
            if d['description_normalized'] == desc_norm
        ),
        'rulebook_conflict_categories': conflict_cats,
        'category_buckets': cat_buckets,
        'cross_bucket_conflict': cross_bucket,
        'file_occurrences': [
            {
                'final_mapping': mapping,
                'bucket': get_bucket(mapping),
                'files': sorted(files),
                'file_count': len(files),
            }
            for mapping, files in sorted(by_mapping.items())
        ],
        'total_files_with_this_row': len(hits),
    })

cross_bucket_entries = [e for e in found_in_files if e['cross_bucket_conflict']]

sources_output = {
    'summary': {
        'total_duplicates_in_rulebook':    len(dup_lookup),
        'duplicates_found_in_xlsm_files':  len(found_in_files),
        'cross_bucket_conflicts':          len(cross_bucket_entries),
        'duplicates_not_seen_in_any_file': len(not_found),
        'xlsm_files_scanned':             len(xlsm_files),
    },
    'conflict_sources':   found_in_files,
    'not_found_in_files': sorted(not_found),
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(sources_output, f, indent=2)

print(f'Results:')
print(f'  {len(found_in_files)} duplicate descriptions found in xlsm files')
print(f'  {len(cross_bucket_entries)} are cross-bucket (REVENUE vs EXPENSE) — most critical')
print(f'  {len(not_found)} duplicates not seen in any file (rulebook-only entries)')
print()

for entry in found_in_files:
    flag = ' CROSS-BUCKET' if entry['cross_bucket_conflict'] else ''
    print(f"  '{entry['description']}'{flag}")
    conflict_str = ' vs '.join(
        f"{c} ({entry['category_buckets'].get(c, '?')})"
        for c in entry['rulebook_conflict_categories']
    )
    print(f'    Conflict: {conflict_str}')
    for occ in entry['file_occurrences']:
        for fname in occ['files']:
            print(f"      {fname}  ->  {occ['final_mapping']} ({occ['bucket']})")
    print()

print(f'Saved: {OUTPUT_JSON}')

## Section 3 — Export to Excel

Converts the conflict sources JSON to a two-tab Excel report.
- **Tab 1 — Category Conflicts:** all descriptions with multiple mappings
- **Tab 2 — Revenue vs Expense:** cross-bucket conflicts only (most critical for NOI)

**Input:** `output_docs/analysis/conflict_file_sources_v2.json` (run Section 2 first)

**Output:** `output_docs/analysis/conflict_sources_report_v2.xlsx`

In [ ]:
INPUT_PATH  = ANALYSIS_OUTPUT_DIR / 'conflict_file_sources_v2.json'
OUTPUT_XLSX = ANALYSIS_OUTPUT_DIR / 'conflict_sources_report_v2.xlsx'

HEADER_FILL = PatternFill('solid', fgColor='1F4E79')
HEADER_FONT = Font(color='FFFFFF', bold=True, size=11)
DESC_FONT   = Font(bold=True, size=10)
CAT_FONT    = Font(size=10)

GROUP_FILLS = [
    PatternFill('solid', fgColor='FFFFFF'),
    PatternFill('solid', fgColor='EBF3FB'),
]

DIVIDER_BORDER = Border(bottom=Side(style='thin', color='BBBBBB'))


def write_tab(ws, entries, tab_title, include_bucket):
    ws.title = tab_title

    headers = ['Row Description', 'Category']
    if include_bucket:
        headers.append('Bucket')
    headers.append('Source Files')

    ws.append(headers)
    for cell in ws[1]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 20

    shade_index = 0

    for entry in entries:
        desc = entry['description']
        fill = GROUP_FILLS[shade_index % 2]
        first_in_group = True

        for occ in entry['file_occurrences']:
            files_str = '\n'.join(occ['files'])

            row_values = [desc if first_in_group else '', occ['final_mapping']]
            if include_bucket:
                row_values.append(occ['bucket'])
            row_values.append(files_str)

            ws.append(row_values)
            r = ws.max_row

            for col in range(1, len(headers) + 1):
                cell = ws.cell(r, col)
                cell.fill = fill
                cell.alignment = Alignment(vertical='top', wrap_text=(col == len(headers)))

            ws.cell(r, 1).font = DESC_FONT if first_in_group else CAT_FONT
            ws.cell(r, 2).font = CAT_FONT

            first_in_group = False

        last_row = ws.max_row
        for col in range(1, len(headers) + 1):
            ws.cell(last_row, col).border = DIVIDER_BORDER

        shade_index += 1

    ws.column_dimensions['A'].width = 38
    ws.column_dimensions['B'].width = 30
    if include_bucket:
        ws.column_dimensions['C'].width = 10
        ws.column_dimensions['D'].width = 55
    else:
        ws.column_dimensions['C'].width = 55

    ws.freeze_panes = 'A2'

In [ ]:
with open(INPUT_PATH) as f:
    data = json.load(f)

all_entries   = data['conflict_sources']
cross_entries = [e for e in all_entries if e['cross_bucket_conflict']]

wb = openpyxl.Workbook()
write_tab(wb.active,         all_entries,   'Category Conflicts', include_bucket=False)
write_tab(wb.create_sheet(), cross_entries, 'Revenue vs Expense', include_bucket=True)

wb.save(OUTPUT_XLSX)

print(f'Saved: {OUTPUT_XLSX}')
print(f"  Tab 1 'Category Conflicts': {len(all_entries)} descriptions")
print(f"  Tab 2 'Revenue vs Expense': {len(cross_entries)} descriptions")

## Section 4 — Rulebook-First Approach Test

Tests whether looking up every row (headers, subtotals, and data rows) directly in the rulebook
is viable as an alternative to block detection.

**Result from initial testing: REJECTED — 27% header conflict rate.**

**Requires:** `output_docs/1_mapping/block_detection/` files (run `detect_blocks.py` first).

In [ ]:
with open(RULEBOOK_DB_V2_PATH) as f:
    rulebook_test = json.load(f)

reverse_map_test = {}
for category, descriptions in rulebook_test.items():
    for desc in descriptions:
        reverse_map_test[desc.strip().lower()] = category

print(f'Rulebook: {len(rulebook_test)} categories, {len(reverse_map_test)} description entries')
print(f'Rulebook path: {RULEBOOK_DB_V2_PATH}\n')

In [ ]:
json_files = sorted(BLOCK_DETECTION_DIR.glob('*.json'))
print(f'Input files ({len(json_files)} total): {BLOCK_DETECTION_DIR}')
for f in json_files:
    print(f'  - {f.name}')
print()

totals    = {'data': 0, 'header': 0, 'subtotal': 0}
found_map = {'data': 0, 'header': 0, 'subtotal': 0}
conflicts = []

for jfile in json_files:
    with open(jfile) as f:
        data = json.load(f)

    fname = data['file']
    fmt   = data['format']

    for row in data['rows']:
        row_type = row['type']
        desc     = row['description'].strip()
        mapping  = reverse_map_test.get(desc.lower())

        totals[row_type] = totals.get(row_type, 0) + 1

        if mapping:
            found_map[row_type] = found_map.get(row_type, 0) + 1
            if row_type in ('header', 'subtotal'):
                conflicts.append({
                    'file':             fname,
                    'format':           fmt,
                    'row_number':       row['row'],
                    'row_type':         row_type,
                    'description':      desc,
                    'rulebook_maps_to': mapping,
                    'section_context':  row.get('section_context', ''),
                })

print('=' * 65)
print('SUMMARY — Rulebook lookup across ALL row types')
print('=' * 65)
for t in ['data', 'header', 'subtotal']:
    total = totals.get(t, 0)
    hit   = found_map.get(t, 0)
    miss  = total - hit
    pct   = (hit / total * 100) if total else 0
    print(f'  {t.upper():<10}  total={total:4}  found={hit:4}  not_found={miss:4}  ({pct:.0f}% hit rate)')

print()
print(f'CONFLICTS (header/subtotal rows that rulebook can map): {len(conflicts)}')
print('These rows would be INCORRECTLY treated as DATA rows under rulebook-first logic.')

if conflicts:
    print()
    print('-' * 65)
    by_file = defaultdict(list)
    for c in conflicts:
        by_file[c['file']].append(c)

    for fname, rows in sorted(by_file.items()):
        print(f"\nFile: {fname}  (format: {rows[0]['format']})")
        for c in rows:
            rn = c['row_number']
            rt = c['row_type'].upper()
            d  = c['description']
            m  = c['rulebook_maps_to']
            print(f'  row {rn:3d}  [{rt:<8}]  {d}  -->  rulebook: {m}')

print()
print('=' * 65)
print('VERDICT')
print('=' * 65)

header_found   = found_map.get('header', 0)
header_total   = totals.get('header', 1)
subtotal_found = found_map.get('subtotal', 0)
subtotal_total = totals.get('subtotal', 1)

header_conflict_pct   = (header_found / header_total) * 100
subtotal_conflict_pct = (subtotal_found / subtotal_total) * 100

print(f'  HEADER   conflict rate: {header_conflict_pct:.0f}%  ({header_found}/{header_total} rows)')
print(f'  SUBTOTAL conflict rate: {subtotal_conflict_pct:.0f}%  ({subtotal_found}/{subtotal_total} rows)')
print()

if header_conflict_pct > 15:
    print('  HIGH header conflict rate — rulebook-first for HEADER rows is NOT safe.')
    print('  Recommendation: keep current detect_blocks classification for headers.')
else:
    print('  Low conflict rate — rulebook-first may be viable.')

if subtotal_conflict_pct > 10:
    print('  HIGH subtotal conflict rate — rulebook-first for SUBTOTAL rows is NOT safe.')
else:
    print('  Low subtotal conflict rate — rulebook-first may be viable for subtotals.')